In [7]:
##### IMPORTS

# General
import re
import numpy as np

from pathlib import Path
from itertools import product

# Other
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [8]:
##### CONFIGURATIONS

outputs = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
out_sin = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs\t1550g4f1k1e9.npz"

In [9]:
# Filename parsing
# Accepts names like: t1450g4.5f2k1e9.npz
# Since k-part is 'k1e9' -> kzz = 1e9
_fname_re = re.compile(r"""^t(?P<Teff>\d+)
                       g(?P<logg>\d+(?:\.\d+)?)
                       f(?P<fsed>\d+(?:\.\d+)?)
                       (?P<ktag>k[0-9eE\+\-\.]+)
                       \.npz$""",
                       re.VERBOSE)

def _parse_kzz(ktag: str) -> float:
    """
    Parse kzz from a tag like 'k9' (-> 1e9) or 'k1e9' (-> 1e9).
    Returns float('nan') if it can't parse.
    """
    s = ktag[1:]  # strip leading 'k'
    # Case 1: integer exponent, e.g., "9" -> 1e9
    if re.fullmatch(r"\d+", s):
        return 10.0 ** int(s)
    # Case 2: scientific or float literal, e.g., "1e9", "1.0e9"
    try:
        return float(s)
    except Exception:
        return float('nan')

def _params_from_filename(p: Path):
    """
    Extract Teff, logg, fsed, kzz from filename.
    """
    m = _fname_re.match(p.name)
    if not m:
        raise ValueError(f"Filename does not match expected pattern: {p.name}")
    Teff = float(m.group("Teff"))
    logg = float(m.group("logg"))
    fsed = float(m.group("fsed"))
    kzz  = _parse_kzz(m.group("ktag"))
    return dict(Teff=Teff, logg=logg, fsed=fsed, kzz=kzz, filename=p.name)

# -------- file IO (MARGE single-case) --------
def _read_npz_perfile(npz_path):
    """
    Return (w_um, F, params dict) from a MARGE per-case .npz:
      - x: wavelength [micron], saved with shape (1, N)
      - y: flux per wavenumber [erg cm^-2 s^-1 (cm^-1)^-1], shape (1, N)
    """
    npz_path = Path(npz_path)
    d = np.load(npz_path, allow_pickle=False)
    # Expect x, y keys with leading axis of size 1
    x = np.asarray(d["x"])
    y = np.asarray(d["y"])

    # Squeeze a possible leading singleton dimension
    if x.ndim == 2 and x.shape[0] == 1:
        x = x[0]
    if y.ndim == 2 and y.shape[0] == 1:
        y = y[0]

    W = x.astype(float)  # wavelength [micron]
    F = y.astype(float)  # flux per wavenumber

    params = _params_from_filename(npz_path)
    return W, F, params

# -------- quick stats --------
def inspect_npz_file(npz_path):
    """
    Print quick stats for a per-case .npz (MARGE format).
    """
    W, F, p = _read_npz_perfile(npz_path)

    print(f"File            : {p['filename']}")
    print(f"Teff [K]        : {p['Teff']:.0f}")
    print(f"logg [cgs]      : {p['logg']:.2f}")
    print(f"f_sed           : {p['fsed']:.2f}")
    print(f"kzz [cm^2 s^-1] : {p['kzz']:.3e}")
    print(f"Wave points     : {W.size} (micron)")
    print(f"λ range         : {W.min():.4f} – {W.max():.4f} μm")
    print(f"F finite?       : {np.isfinite(F).all()}")
    print(f"F min/max       : {np.nanmin(F):.3e} / {np.nanmax(F):.3e}")

    return dict(wavelength_um=W, F=F, **p)

# -------- plotting --------
def plot_npz_file(npz_path, logy=False):
    """
    Plot a single .npz (MARGE format).
    """
    W, F, p = _read_npz_perfile(npz_path)

    title = (f"{p['filename']} — "
             f"Teff={p['Teff']:.0f}, logg={p['logg']:.2f}, "
             f"f_sed={p['fsed']:.2f}, kzz={p['kzz']:.2e}")
    src = ColumnDataSource(dict(wavelength_um=W, F=F))

    tools = "pan,wheel_zoom,box_zoom,reset,save"
    fig = figure(title=title,
                 x_axis_label="Wavelength (μm)",
                 y_axis_label="F (erg cm⁻² s⁻¹ cm⁻¹)",
                 sizing_mode="stretch_width", height=420,
                 tools=tools, output_backend="canvas")
    if logy:
        fig.y_axis_type = "log"

    fig.add_tools(HoverTool(tooltips=[("λ (μm)", "@wavelength_um{0.000}"),
                                      ("F", "@F{0.00e}")],
                            mode="vline"))
    fig.line('wavelength_um', 'F', source=src, line_width=2)
    show(fig)

In [10]:
inspect_npz_file(out_sin)
plot_npz_file(out_sin, logy=False)

File            : t1550g4f1k1e9.npz
Teff [K]        : 1550
logg [cgs]      : 4.00
f_sed           : 1.00
kzz [cm^2 s^-1] : 1.000e+09
Wave points     : 844 (micron)
λ range         : 0.3005 – 4.9914 μm
F finite?       : True
F min/max       : 3.934e+05 / 8.055e+11
